# 2.3 — Tối ưu MLP và xử lý mất cân bằng

So sánh hai thay đổi có kiểm soát với MLP task 2.1: BCEWithLogitsLoss có pos_weight tính từ train; và MLP dùng embeddings cho các cột phân loại. Cả hai giữ đúng membership validation seed 9. Không resample validation, không đọc test labels.

Đầu ra gồm bảng thí nghiệm, xác suất validation, checkpoint và learning curves. Chọn mô hình theo validation ROC-AUC; Average Precision là metric phụ.

In [ ]:
import json
import random
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

SEED = 9
TARGET = "is_fraud"
ID_COL = "id"
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e11")
OUTPUT_DIR = Path("/kaggle/working/task_2_3_mlp")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2.3.1–2.3.2 — Khôi phục split và baseline 2.1

Nạp split membership và kết quả baseline từ output task 2.1 đã gắn làm input. Ghép theo id, kiểm tra số dòng và tính class weight chỉ từ train.

In [ ]:
input_root = Path("/kaggle/input")
split_files = list(input_root.rglob("split_ids_seed9.csv"))
if len(split_files) != 1:
    raise FileNotFoundError("Gắn output task 2.1 để nạp duy nhất split_ids_seed9.csv.")
split_ids = pd.read_csv(split_files[0])
train = pd.read_csv(DATA_DIR / "train.csv")
train = train.merge(split_ids[[ID_COL, "split"]], on=ID_COL, how="left", validate="one_to_one", sort=False)
if train["split"].isna().any():
    raise ValueError("Có train row không có membership trong split seed 9.")

train_part = train.loc[train["split"].eq("train")].copy()
valid_part = train.loc[train["split"].eq("validation")].copy()
y_train = train_part[TARGET].astype(np.float32).to_numpy()
y_valid = valid_part[TARGET].astype(np.int64).to_numpy()
positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
pos_weight = negative_count / positive_count

X_train_raw = train_part.drop(columns=[ID_COL, TARGET, "split"]).copy()
X_valid_raw = valid_part.drop(columns=[ID_COL, TARGET, "split"]).copy()
bool_cols = X_train_raw.select_dtypes(include=["bool"]).columns.tolist()
for frame in (X_train_raw, X_valid_raw):
    frame[bool_cols] = frame[bool_cols].astype("int8")
cat_cols = X_train_raw.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [col for col in X_train_raw.columns if col not in cat_cols]
onehot_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), num_cols),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cat_cols),
    ],
    remainder="drop",
)
X_train = onehot_preprocessor.fit_transform(X_train_raw).astype(np.float32)
X_valid = onehot_preprocessor.transform(X_valid_raw).astype(np.float32)
joblib.dump(onehot_preprocessor, OUTPUT_DIR / "onehot_preprocessor.joblib")

metric_paths = list(input_root.rglob("metrics.json"))
baseline_metrics = None
for path in metric_paths:
    try:
        candidate = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if candidate.get("task") == "2.1 MLP baseline":
        baseline_metrics = candidate
        break
if baseline_metrics is None:
    raise FileNotFoundError("Không tìm thấy metrics.json của task 2.1.")
baseline_prediction_paths = list(input_root.rglob("validation_predictions_seed9.csv"))
if len(baseline_prediction_paths) != 1:
    raise FileNotFoundError("Cần duy nhất validation_predictions_seed9.csv từ task 2.1.")
baseline_predictions = pd.read_csv(baseline_prediction_paths[0])
expected_ids = valid_part[ID_COL].to_numpy()
if baseline_predictions[ID_COL].duplicated().any() or set(baseline_predictions[ID_COL]) != set(expected_ids):
    raise ValueError("Task 2.1 prediction IDs do not match validation membership.")
baseline_predictions = baseline_predictions.set_index(ID_COL).loc[expected_ids].reset_index()
print("Rows train/validation:", len(train_part), len(valid_part))
print("Fraud rate in train:", round(float(y_train.mean()), 6))
print("Train class counts:", {"legitimate": negative_count, "fraud": positive_count})
print("pos_weight = n_negative / n_positive:", round(pos_weight, 4))
print("Features:", {"numeric_and_boolean": len(num_cols), "categorical": len(cat_cols), "one-hot total": X_train.shape[1]})
print("Baseline 2.1 ROC-AUC/AP:", baseline_metrics["validation_roc_auc"], baseline_metrics["validation_average_precision"])

## 2.3.3 — MLP có trọng số cho lớp fraud

pos_weight được tính bằng số mẫu không fraud chia cho số mẫu fraud trong train. Nó tăng mức phạt cho lỗi bỏ sót fraud, nhưng không thay đổi validation hoặc phân phối dữ liệu. Kiến trúc, preprocessing, optimizer và giới hạn epoch giữ giống baseline để tập trung đo ảnh hưởng của weighting.

In [ ]:
class DenseMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x)


def train_experiment(name, model, train_features, valid_features, weighting=None):
    start_time = time.time()
    model = model.to(DEVICE)
    train_tensors = [torch.from_numpy(np.asarray(x, dtype=np.int64 if np.issubdtype(np.asarray(x).dtype, np.integer) else np.float32)) for x in train_features]
    target_tensor = torch.from_numpy(y_train.reshape(-1, 1))
    loader = DataLoader(
        TensorDataset(*train_tensors, target_tensor),
        batch_size=4096,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=torch.Generator().manual_seed(SEED),
    )
    valid_tensors = [torch.from_numpy(np.asarray(x, dtype=np.int64 if np.issubdtype(np.asarray(x).dtype, np.integer) else np.float32)).to(DEVICE) for x in valid_features]
    criterion = nn.BCEWithLogitsLoss() if weighting is None else nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([weighting], dtype=torch.float32, device=DEVICE)
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auc, best_epoch, stale_epochs, best_state = -np.inf, 0, 0, None
    history = []
    for epoch in range(1, 13):
        epoch_start = time.time()
        model.train()
        loss_sum, seen = 0.0, 0
        for batch in loader:
            *batch_features, batch_y = batch
            batch_features = [x.to(DEVICE, non_blocking=True) for x in batch_features]
            batch_y = batch_y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(*batch_features), batch_y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            loss_sum += float(loss.item()) * len(batch_y)
            seen += len(batch_y)
        model.eval()
        with torch.inference_mode():
            probability = torch.sigmoid(model(*valid_tensors).squeeze(1)).cpu().numpy()
        val_auc = float(roc_auc_score(y_valid, probability))
        val_ap = float(average_precision_score(y_valid, probability))
        train_loss = loss_sum / seen
        history.append({"epoch": epoch, "train_loss": train_loss, "val_roc_auc": val_auc, "val_average_precision": val_ap})
        print(f"{name} | epoch {epoch:02d} | loss={train_loss:.5f} | ROC-AUC={val_auc:.6f} | AP={val_ap:.6f} | {time.time()-epoch_start:.1f}s")
        if val_auc > best_auc + 1e-6:
            best_auc, best_epoch, stale_epochs = val_auc, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale_epochs += 1
            if stale_epochs >= 3:
                print(f"{name}: early stopping; best epoch={best_epoch}.")
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.inference_mode():
        probability = torch.sigmoid(model(*valid_tensors).squeeze(1)).cpu().numpy().astype(np.float32)
    safe_name = name.lower().replace(" ", "_")
    model_dir = OUTPUT_DIR / safe_name
    model_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        ID_COL: valid_part[ID_COL].to_numpy(),
        TARGET: y_valid,
        "fraud_probability": probability,
    }).to_csv(model_dir / f"validation_predictions_{safe_name}_seed9.csv", index=False)
    history_df = pd.DataFrame(history)
    history_df.to_csv(model_dir / "training_history.csv", index=False)
    torch.save(
        {"state_dict": best_state, "seed": SEED, "best_epoch": best_epoch, "pos_weight": weighting},
        model_dir / "best_model.pt",
    )
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o")
    axes[0].set(title="Training loss", xlabel="Epoch", ylabel="BCE loss")
    axes[1].plot(history_df["epoch"], history_df["val_roc_auc"], marker="o")
    axes[1].set(title="Validation ROC-AUC", xlabel="Epoch", ylabel="ROC-AUC")
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(model_dir / "learning_curves.png", dpi=150)
    plt.show()
    plt.close(fig)

    result = {
        "name": name,
        "training_seconds": float(time.time() - start_time),
        "validation_roc_auc": float(roc_auc_score(y_valid, probability)),
        "validation_average_precision": float(average_precision_score(y_valid, probability)),
        "best_epoch": best_epoch,
        "epochs_run": len(history),
        "pos_weight": weighting,
    }
    with open(model_dir / "metrics.json", "w", encoding="utf-8") as file:
        json.dump(result, file, indent=2)
    return result

## 2.3.4 — Chạy MLP weighted

Chạy một lượt MLP có weighting trên one-hot features. Early stopping theo ROC-AUC giới hạn số epoch; lưu checkpoint, dự đoán validation, thời gian train và learning curves.

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

weighted_result = train_experiment(
    "MLP weighted",
    DenseMLP(X_train.shape[1]),
    [X_train],
    [X_valid],
    weighting=pos_weight,
)

## 2.3.5 — MLP với embeddings cho category

Thử một biến thể phù hợp schema: thay one-hot bằng embeddings cho các cột phân loại, giữ numeric/boolean đã chuẩn hóa, cùng seed, optimizer và loss không trọng số. Category chưa thấy trong train được mã hóa thành 0.

In [ ]:
X_train_num_raw = train_part[num_cols].copy()
X_valid_num_raw = valid_part[num_cols].copy()
for frame in (X_train_num_raw, X_valid_num_raw):
    for col in num_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
numeric_medians = X_train_num_raw.median()
X_train_num_raw = X_train_num_raw.fillna(numeric_medians)
X_valid_num_raw = X_valid_num_raw.fillna(numeric_medians)
embedding_scaler = StandardScaler()
X_train_num = embedding_scaler.fit_transform(X_train_num_raw).astype(np.float32)
X_valid_num = embedding_scaler.transform(X_valid_num_raw).astype(np.float32)

X_train_cat = np.zeros((len(train_part), len(cat_cols)), dtype=np.int64)
X_valid_cat = np.zeros((len(valid_part), len(cat_cols)), dtype=np.int64)
category_mappings, cat_dims, cat_emb_dims = {}, [], []
for cat_idx, col in enumerate(cat_cols):
    train_values = train_part[col].astype("string").fillna("__MISSING__")
    valid_values = valid_part[col].astype("string").fillna("__MISSING__")
    categories = sorted(train_values.astype(str).unique().tolist())
    mapping = {value: index + 1 for index, value in enumerate(categories)}
    category_mappings[col] = mapping
    X_train_cat[:, cat_idx] = train_values.astype(str).map(mapping).to_numpy(dtype=np.int64)
    X_valid_cat[:, cat_idx] = valid_values.astype(str).map(mapping).fillna(0).to_numpy(dtype=np.int64)
    cat_dims.append(len(mapping) + 1)
    cat_emb_dims.append(min(16, max(2, int(np.ceil(np.sqrt(len(mapping) + 1))))))

class EmbeddingMLP(nn.Module):
    def __init__(self, n_numeric, dimensions, embedding_dimensions):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality, embedding_dim in zip(dimensions, embedding_dimensions)
        ])
        self.network = nn.Sequential(
            nn.Linear(n_numeric + sum(embedding_dimensions), 128), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.10), nn.Linear(64, 1),
        )

    def forward(self, numeric, categorical):
        embedded = [layer(categorical[:, i]) for i, layer in enumerate(self.embeddings)]
        return self.network(torch.cat([numeric, *embedded], dim=1))

joblib.dump(
    {
        "scaler": embedding_scaler,
        "numeric_medians": numeric_medians.to_dict(),
        "category_mappings": category_mappings,
        "numeric_columns": num_cols,
        "categorical_columns": cat_cols,
        "categorical_dimensions": cat_dims,
        "embedding_dimensions": cat_emb_dims,
        "seed": SEED,
    },
    OUTPUT_DIR / "embedding_preprocessor.joblib",
)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
embedding_result = train_experiment(
    "MLP embedding",
    EmbeddingMLP(len(num_cols), cat_dims, cat_emb_dims),
    [X_train_num, X_train_cat.astype(np.int64)],
    [X_valid_num, X_valid_cat.astype(np.int64)],
)

## 2.3.6–2.3.7 — Bảng thí nghiệm và lưu kết quả

Đưa baseline 2.1 cùng hai lượt thử vào một bảng để so sánh ROC-AUC, Average Precision, thời gian train và số epoch. Chọn theo validation ROC-AUC; PR-AUC là metric phụ. Không sử dụng kết quả public leaderboard để chọn cấu hình.

In [ ]:
experiment_table = pd.DataFrame([
    {
        "model": "MLP baseline",
        "validation_roc_auc": baseline_metrics["validation_roc_auc"],
        "validation_average_precision": baseline_metrics["validation_average_precision"],
        "training_seconds": None,
        "best_epoch": baseline_metrics["best_epoch"],
    },
    {
        "model": weighted_result["name"],
        "validation_roc_auc": weighted_result["validation_roc_auc"],
        "validation_average_precision": weighted_result["validation_average_precision"],
        "training_seconds": weighted_result["training_seconds"],
        "best_epoch": weighted_result["best_epoch"],
    },
    {
        "model": embedding_result["name"],
        "validation_roc_auc": embedding_result["validation_roc_auc"],
        "validation_average_precision": embedding_result["validation_average_precision"],
        "training_seconds": embedding_result["training_seconds"],
        "best_epoch": embedding_result["best_epoch"],
    },
]).sort_values(["validation_roc_auc", "validation_average_precision"], ascending=False).reset_index(drop=True)

experiment_table.to_csv(OUTPUT_DIR / "experiment_table.csv", index=False)
summary = {
    "task": "2.3 MLP imbalance experiments",
    "seed": SEED,
    "split": "shared stratified 80/20 split from task 2.1",
    "pos_weight_formula": "n_negative / n_positive from training fold",
    "positive_count_train": positive_count,
    "negative_count_train": negative_count,
    "pos_weight": float(pos_weight),
    "experiments": [baseline_metrics, weighted_result, embedding_result],
    "selected_by_validation_roc_auc": str(experiment_table.iloc[0]["model"]),
}
with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

display(experiment_table)
print("Saved experiment artifacts to:", OUTPUT_DIR)